# Objetivo

Desenvolvimento do Teleutils em ambiente de teste local

# Configuração do Ambiente

## Bibliotecas

In [ ]:
# 1. Módulos da Biblioteca Padrão
import logging
from pathlib import Path

# 2. Módulos de Terceiros
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import pandas_udf
from pyspark.sql import types as T # ArrayType, StructType, StructField, StringType, BooleanType

# 3. Teletools/Teleutils
from teleutils.core.extractors import CDRParquetExtractor, CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

In [ ]:
# Configuração do Pandas
# Exibir todas as colunas
pd.set_option("display.max_columns", None)

# Exibir todas as linhas
pd.set_option("display.max_rows", None)

# Evitar que o texto nas células seja cortado
pd.set_option("display.max_colwidth", None)

In [ ]:
# Configuração para exibir logs no output da célula
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)

# Reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark

In [ ]:
spark = (
    SparkSession
        .builder
        .appName("SessaoLocal")
        .master("local[12]")
        .config("spark.driver.memory", "24g")
        .getOrCreate()
)
spark

## Arquivos de dados

### Dados de origem

In [ ]:
folder_vivo = Path("/assets/monitoring/cdr/cdr_bruto/Semana101/Vivo/FIXA")
files_vivo = [str(file) for file in folder_vivo.glob("*.gz")]
files_vivo = files_vivo[:2]
files_vivo

### Arquivos intermediário e final

In [ ]:
extracted_vivo = "/assets/data/cdr/teletools/db/extraidos/fcdr_vivo_extraido.parquet"
transformed_vivo = "/assets/data/cdr/teletools/db/transformados/fcdr_vivo_transformado.parquet"

# Análise

## Dados brutos

In [ ]:
df = spark.read.csv(files_vivo, sep=";")
df.limit(5).toPandas()

## Extração

In [ ]:
extractor = CDRTextExtractor(spark)
extractor.extract(files_vivo, extracted_vivo, "stfc_fcdr_vivo")

In [ ]:
df_extracted = spark.read.parquet(extracted_vivo)
df_extracted.show(5)

## Transformação

In [ ]:
transformer = CDRTransformer(spark)
transformer.transform_stfc_fcdr_vivo(extracted_vivo, transformed_vivo)

In [ ]:
df_transformed = spark.read.parquet(transformed_vivo)
df_transformed.limit(5).toPandas().dropna(how="all",axis=1)